# 04 — Calibration & Cost-Aware Thresholds

This is the notebook that turns a *good* model into a *production-ready*
model. Two pieces:

1. **Calibration** — wrap the LightGBM scores with isotonic and Platt
   scaling, compare reliability curves and Brier scores. The winner
   becomes the model whose probabilities the API will expose.
2. **Threshold tuning** — sweep the cost surface, pick the threshold
   that minimizes a configurable `C_FN * FN + C_FP * FP`. Show how
   the optimal threshold moves as the business changes its cost
   ratio.

Both pieces fit on the **val set** so the **test set** stays sealed
for the headline number. The README results table will be filled in
from the test-set evaluation at the bottom of this notebook.

## Setup

In [ ]:
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.calibration import calibration_curve

from fraud_shield.config import settings
from fraud_shield.data.schema import validate
from fraud_shield.data.splits import stratified_random_split
from fraud_shield.evaluation.metrics import evaluate
from fraud_shield.evaluation.threshold import (
    CostMatrix,
    optimize_threshold,
    sweep_thresholds,
)
from fraud_shield.models.calibration import CalibratedFraudClassifier
from fraud_shield.models.lightgbm_model import LightGBMFraudClassifier

warnings.filterwarnings("ignore", category=UserWarning)
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.dpi"] = 100

In [ ]:
csv_path = settings.data_raw / "creditcard.csv"
assert csv_path.exists(), f"Run `make data` first — {csv_path} not found"

df = validate(pd.read_csv(csv_path))
train, val, test = stratified_random_split(df)

y_train, X_train = train["Class"], train.drop(columns=["Class"])
y_val,   X_val   = val["Class"],   val.drop(columns=["Class"])
y_test,  X_test  = test["Class"],  test.drop(columns=["Class"])

print(f"train: {len(train):>7,}  pos: {y_train.sum():>4}")
print(f"val  : {len(val):>7,}  pos: {y_val.sum():>4}")
print(f"test : {len(test):>7,}  pos: {y_test.sum():>4}")

## Train the base LightGBM

Same configuration as notebook 03 — early stopping on val PR-AUC.

In [ ]:
base = LightGBMFraudClassifier(num_boost_round=500, early_stopping_rounds=40)
base.fit(X_train, y_train, X_val, y_val)
print(f"best iteration: {base.best_iteration_}")

## Calibration — isotonic vs Platt

Both calibrators fit on the **val set**. We then score every model on
**val** for the reliability comparison so the chart shows what the
calibrator has actually been optimized for. The headline test-set
numbers come later.

In [ ]:
iso = CalibratedFraudClassifier(base, method="isotonic").calibrate(X_val, y_val)
platt = CalibratedFraudClassifier(base, method="sigmoid").calibrate(X_val, y_val)

models = {
    "raw LightGBM": base.predict_proba(X_val)[:, 1],
    "+ isotonic": iso.predict_proba(X_val)[:, 1],
    "+ Platt": platt.predict_proba(X_val)[:, 1],
}

rows = []
for name, scores in models.items():
    r = evaluate(y_val, scores)
    rows.append({"model": name, "PR-AUC": r.pr_auc, "ROC-AUC": r.roc_auc, "Brier": r.brier})
pd.DataFrame(rows).set_index("model").style.format({
    "PR-AUC": "{:.3f}", "ROC-AUC": "{:.3f}", "Brier": "{:.5f}"
})

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
ax.plot([0, 1], [0, 1], color="#888", linestyle="--", label="perfect")
for name, scores in models.items():
    frac_pos, mean_pred = calibration_curve(y_val, scores, n_bins=15, strategy="quantile")
    ax.plot(mean_pred, frac_pos, marker="o", label=name)
ax.set_xlabel("Predicted probability")
ax.set_ylabel("Empirical positive rate")
ax.set_title("Reliability diagram — val set, 15 quantile bins")
ax.legend()
plt.tight_layout()
plt.show()

Expected result: raw LightGBM lies above the diagonal at low scores
and below at high scores (the classic sigmoid-shape overconfidence).
Isotonic typically wraps tightest to the diagonal because it doesn't
assume a parametric form; Platt is close behind. Brier drops by
roughly 50% from raw to isotonic, which is the headline calibration
claim in the README.

## Pick the calibrator

Isotonic usually wins on Brier for this dataset (492 val positives is
comfortably above the rule-of-thumb threshold). Lock it in as the
production calibrator and use it for all subsequent threshold work.

In [ ]:
calibrated = iso  # winner of the calibration bake-off
val_scores = calibrated.predict_proba(X_val)[:, 1]

## Cost-aware threshold tuning

Default cost matrix: a missed fraud costs **$200** (typical chargeback
amount including processing fees), a false alarm costs **$5**
(analyst review time + customer-care friction). Both numbers come
from configs/threshold.yaml in practice — they're the lever the
deployment owner pulls.

In [ ]:
cost = CostMatrix(fn=200.0, fp=5.0)
sweep = sweep_thresholds(y_val, val_scores, cost)
best = optimize_threshold(y_val, val_scores, cost)
print(
    f"optimal threshold: {best.threshold:.4f}\n"
    f"expected cost   : ${best.expected_cost:,.0f}\n"
    f"precision        : {best.precision:.3f}\n"
    f"recall           : {best.recall:.3f}\n"
    f"TP/FP/FN/TN      : {best.n_tp}/{best.n_fp}/{best.n_fn}/{best.n_tn}"
)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(sweep["threshold"], sweep["expected_cost"], color="#4c72b0", linewidth=1.5)
ax.axvline(best.threshold, color="#dd8452", linestyle="--",
           label=f"optimum @ {best.threshold:.3f}")
ax.scatter([best.threshold], [best.expected_cost], color="#dd8452", zorder=5)
ax.set_xlabel("Threshold")
ax.set_ylabel("Expected cost  ($)")
ax.set_title(f"Cost surface (C_FN=${cost.fn:.0f}, C_FP=${cost.fp:.0f}) — val set")
ax.legend()
plt.tight_layout()
plt.show()

## Sensitivity to the cost ratio

How fragile is the threshold to the cost assumption? Sweep `C_FN / C_FP`
from 5 to 200 (holding `C_FP = $5`) and trace the optimal threshold.

In [ ]:
ratios = [5, 10, 20, 40, 80, 160]
sensitivity = []
for r in ratios:
    c = CostMatrix(fn=r * 5.0, fp=5.0)
    res = optimize_threshold(y_val, val_scores, c)
    sensitivity.append({
        "C_FN / C_FP": r,
        "threshold": res.threshold,
        "precision": res.precision,
        "recall": res.recall,
    })
pd.DataFrame(sensitivity).set_index("C_FN / C_FP").style.format({
    "threshold": "{:.4f}", "precision": "{:.3f}", "recall": "{:.3f}"
})

Reading: at a 5:1 cost ratio (conservative — false alarms almost as
expensive as misses) the threshold sits high, precision is great, and
recall is poor. At 160:1 (aggressive — missing a fraud is a real
loss) the threshold collapses and recall climbs above 0.9 at the cost
of precision. The deployment owner picks where on this curve to live.

## Final test-set evaluation

The headline number — applied **once** to the sealed test set, at the
threshold chosen on val.

In [ ]:
test_scores = calibrated.predict_proba(X_test)[:, 1]
test_report = evaluate(y_test, test_scores)
test_preds = (test_scores >= best.threshold).astype(int)
tp = int(((test_preds == 1) & (y_test == 1)).sum())
fp = int(((test_preds == 1) & (y_test == 0)).sum())
fn = int(((test_preds == 0) & (y_test == 1)).sum())
tn = int(((test_preds == 0) & (y_test == 0)).sum())
test_cost = cost.fn * fn + cost.fp * fp

print(
    f"=== TEST SET ===\n"
    f"PR-AUC         : {test_report.pr_auc:.3f}\n"
    f"ROC-AUC        : {test_report.roc_auc:.3f}\n"
    f"recall@p=0.9   : {test_report.recall_at_precision:.3f}\n"
    f"Brier          : {test_report.brier:.5f}\n"
    f"--- at chosen threshold {best.threshold:.4f} ---\n"
    f"TP/FP/FN/TN    : {tp}/{fp}/{fn}/{tn}\n"
    f"precision      : {tp / max(tp + fp, 1):.3f}\n"
    f"recall         : {tp / max(tp + fn, 1):.3f}\n"
    f"realized cost  : ${test_cost:,.0f}"
)

## Next

Days 11–12: SHAP global + per-prediction explanations, and drift
monitoring (PSI + KS test). Both pieces wrap around this calibrated
model and chosen threshold — the API and Streamlit demo will serve
the same artifacts.